### 3.8.2. Branch and Bound

$$
z^\star = \min_{\mathbf{x}\in\mathcal{X}\cap\mathbb{Z}^n} \mathbf{c}^\top\mathbf{x},
\qquad
\underline{z}(\text{node}) \le z^\star \le \overline{z}\ (\text{incumbent}).
$$

**Explanation:**

Branch and bound solves nonconvex and integer problems by recursively splitting the feasible set (*branching*) and computing a bound on the best value achievable in each piece (*bounding*), pruning any branch whose bound cannot beat the best feasible solution found so far (the *incumbent*). For integer programs the bound comes from the [LP relaxation](../04_Convex_Problem_Classes/01_linear_programming.ipynb): if a node's relaxed optimum is already worse than the incumbent, or is integer-feasible, the node is closed. The method is exact and terminates because the branching tree is finite.

**Intuition:**

Each split fixes a fractional variable up or down; branches whose optimistic bound is dominated are discarded without full exploration.

![Branch-and-bound tree for the integer program](../../Figures/030802_branch_and_bound_tree.png)

**Numerical Example:**

$$
\max\; 5x_1 + 4x_2
\quad\text{s.t.}\quad 6x_1 + 4x_2 \le 24,\; x_1 + 2x_2 \le 6,\; x_1,x_2\in\mathbb{Z}_{\ge0}.
$$

The root LP relaxation gives $21.0$ at $(3, 1.5)$. Branching on $x_2$: the branch $x_2\ge 2$ becomes infeasible for integer $x_1$ near the bound, while $x_2\le 1$ leads through $x_1\le 3$ (value $19$) and $x_1\ge 4$ to the integer point $(4,0)$ with value $20$. No open branch exceeds $20$, so the incumbent $(4,0)$ with value $20$ is proved optimal.

In [1]:
import math
from scipy.optimize import linprog

def relaxation_value(extra_bounds):
    bounds = [extra_bounds.get(0, (0, None)), extra_bounds.get(1, (0, None))]
    result = linprog(c=[-5, -4], A_ub=[[6, 4], [1, 2]], b_ub=[24, 6], bounds=bounds)
    return (None if not result.success else (-result.fun, result.x))

incumbent = (-math.inf, None)
stack = [{}]
while stack:
    node = stack.pop()
    bound = relaxation_value(node)
    if bound is None or bound[0] <= incumbent[0]:
        continue
    value, point = bound
    fractional = next((index for index in (0, 1) if abs(point[index] - round(point[index])) > 1e-6), None)
    if fractional is None:
        incumbent = max(incumbent, (round(value), tuple(int(round(v)) for v in point)), key=lambda item: item[0])
        continue
    floor_value = math.floor(point[fractional])
    stack.append({**node, fractional: (0, floor_value)})
    stack.append({**node, fractional: (floor_value + 1, None)})

print("optimal integer value:", incumbent[0], " at", incumbent[1])

optimal integer value: 20  at (4, 0)


**Equivalent scipy.milp implementation:**

In [2]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

result = milp(c=np.array([-5.0, -4.0]),
              constraints=LinearConstraint(np.array([[6.0, 4.0], [1.0, 2.0]]), -np.inf, [24, 6]),
              integrality=np.ones(2), bounds=Bounds(lb=0, ub=np.inf))

print("optimal integer value:", -result.fun, " at", result.x)

optimal integer value: 20.0  at [ 4. -0.]


**References:**

[📘 Wolsey, L. A. (1998). *Integer Programming*. Wiley.](https://www.wiley.com/en-us/Integer+Programming-p-9781119606536)  
[📘 Horst, R., & Tuy, H. (1996). *Global Optimization: Deterministic Approaches* (3rd ed.). Springer.](https://doi.org/10.1007/978-3-662-03199-5)

---

[⬅️ Previous: Global Optimization](./01_global_optimization.ipynb) | [Next: Bayesian Optimization ➡️](./03_bayesian_optimization.ipynb)